In [1]:
import pandas as pd
import re

# Read Excel
file_path = "updated_dataset.xlsx"
df = pd.read_excel(file_path)


In [2]:
display(df["Top shape"].value_counts())

Top shape
Flat        162
Curved      149
Scrolled    128
Hospital      3
unknown       2
Name: count, dtype: int64

In [3]:
# Fill missing values
df["Name"] = df["Name"].fillna("").astype(str)
df["Tags"] = df["Tags"].fillna("").astype(str)

# Combine text
combined_text = (df["Name"] + " " + df["Tags"]).str.lower()

In [4]:
# word → number mapping
word_to_num = {
    "one":1,"two":2,"three":3,"four":4,"five":5,
    "six":6,"seven":7,"eight":8,"nine":9,"ten":10
}

# function to extract number before keyword
def extract_number(text, keyword):
    pattern = rf'(\d+|one|two|three|four|five|six|seven|eight|nine|ten)\s*{keyword}'
    match = re.search(pattern, text, re.IGNORECASE)
    
    if match:
        value = match.group(1).lower()
        return int(value) if value.isdigit() else word_to_num.get(value)
    return ""

# Extract Column / Row
df["Column"] = combined_text.apply(lambda x: extract_number(x, "column"))
df["Row"] = combined_text.apply(lambda x: extract_number(x, "row"))

In [5]:
# -------- keyword classification --------

keyword_map = {
    "Wall": r'wall',
    "Window": r'window',
    "Ornate": r'decorated|ornamental|ornate|ornimental|ribbon|eared',
    "Scroll": r'scroll',
    "Round top": r'round top',
    "Flat top": r'flat top',
    "Flue": r'flue',
    "Easy Clean": r'easy clean'
}

for col, pattern in keyword_map.items():
    df[col] = combined_text.str.contains(pattern, regex=True).astype(int)

# -------- plain (exclude ideal plain) --------

is_plain = combined_text.str.contains(r'\bplain\b', regex=True)
is_ideal_plain = combined_text.str.contains(r'ideal\s+plain', regex=True)

df["Plain"] = (is_plain & ~is_ideal_plain).astype(int)

# -------- nipple ratio --------

df["Nipple Top / Bottom"] = df['Nipple Size Top (")'] / df['Nipple Size Bottom (")']

# preview
df.head()

,Products ID,Name,Castrads SKU,Family,Manufacturer,Tags,Section Length (mm),Leg Section Depth (mm),Mid Section Depth (mm),Leg Section Height (mm),...,Wall,Window,Ornate,Scroll,Round top,Flat top,Flue,Easy Clean,Plain,Nipple Top / Bottom
0,10670,A. A. Griffing Iron Co Bundy Standard 1 Row 24in,ORC-AAGIC-B-1R24,Bundy,A. A. Griffing Iron Co.,"A. A. Griffing Iron Co,Bundy, Standard, 1 Row,...",76,140,140,610,...,0,0,0,0,0,0,0,0,0,1.0
1,10673,A. A. Griffing Iron Co Bundy Standard 1 Row 27in,ORC-AAGIC-B-1R27,Bundy,A. A. Griffing Iron Co.,"A. A. Griffing Iron Co,Bundy, Standard, 1 Row,...",76,140,140,686,...,0,0,0,0,0,0,0,0,0,1.0
2,10676,A. A. Griffing Iron Co Bundy Standard 1 Row 30in,ORC-AAGIC-B-1R30,Bundy,A. A. Griffing Iron Co.,"A. A. Griffing Iron Co,Bundy, Standard, 1 Row,...",76,140,140,762,...,0,0,0,0,0,0,0,0,0,1.0
3,10679,A. A. Griffing Iron Co Bundy Standard 1 Row 33in,ORC-AAGIC-B-1R33,Bundy,A. A. Griffing Iron Co.,"A. A. Griffing Iron Co,Bundy, Standard, 1 Row,...",76,140,140,838,...,0,0,0,0,0,0,0,0,0,1.0
4,10682,A. A. Griffing Iron Co Bundy Standard 1 Row 36in,ORC-AAGIC-B-1R36,Bundy,A. A. Griffing Iron Co.,"A. A. Griffing Iron Co,Bundy, Standard, 1 Row,...",76,140,140,914,...,0,0,0,0,0,0,0,0,0,1.0


In [6]:
print(df.columns)

Index(['Products ID', 'Name', 'Castrads SKU', 'Family', 'Manufacturer', 'Tags',
       'Section Length (mm)', 'Leg Section Depth (mm)',
       'Mid Section Depth (mm)', 'Leg Section Height (mm)',
       'Mid Section Height (mm)', 'Leg Section Weight (kg)',
       'Mid Section Weight (kg)', 'Internal Volume (L)', 'Nipple Size Top (")',
       'Nipple Size Bottom (")', 'Pipe Centre Top To Floor (mm)',
       'Pipe Centre Bottom To Floor (mm)', 'Inter Axis Distance (mm)',
       'Exponent N', 'Factor Km', 'Output Dt50 (W)', 'Cross-brace type',
       'Top shape', 'Link to google drive image', 'DOP No.', 'Comments',
       'Column', 'Row', 'Wall', 'Window', 'Ornate', 'Scroll', 'Round top',
       'Flat top', 'Flue', 'Easy Clean', 'Plain', 'Nipple Top / Bottom'],
      dtype='object')


In [7]:
print(df["Top shape"].value_counts())

Top shape
Flat        162
Curved      149
Scrolled    128
Hospital      3
unknown       2
Name: count, dtype: int64


In [8]:
df.loc[df["Top shape"] == "Curved", "Round top"] = 1

In [9]:
df.loc[df["Top shape"] == "Flat", "Flat top"] = 1

In [10]:
df.loc[df["Top shape"] == "Scrolled", "Scroll"] = 1

In [11]:
df.loc[df["Top shape"] == "Plain", "Plain"] = 1

In [12]:
cols = ["Round top", "Flat top", "Scroll", "Plain"]

count = (df[cols] == 0).all(axis=1).sum()

print(count)

5


In [13]:
# get rows where all 4 columns are 0
rows = df[(df[cols] == 0).all(axis=1)]

# print the rows
display(rows)

,Products ID,Name,Castrads SKU,Family,Manufacturer,Tags,Section Length (mm),Leg Section Depth (mm),Mid Section Depth (mm),Leg Section Height (mm),...,Wall,Window,Ornate,Scroll,Round top,Flat top,Flue,Easy Clean,Plain,Nipple Top / Bottom
237,10790,Beeston Hospital Wide 30in - 1957,ORC-B-H-W30,Beeston Hospital,Beeston,"Beeston, Hospital, School, Wide",64,178,178,762,...,0,0,0,0,0,0,0,0,0,1.0
239,10793,Beeston Hospital Wide 36in - 1957,ORC-B-H-W36,Beeston Hospital,Beeston,"Beeston, Hospital, Easy Clean, School, Narrow",64,178,178,914,...,0,0,0,0,0,0,0,1,0,1.0
309,12672,Biasi LBT 4 Column 680mm,ORC-BI-4C680,LBT,Biasi,"Column Radiators, Four Column Radiators",60,146,146,680,...,0,0,0,0,0,0,0,0,0,1.0
310,12669,Biasi LBT 6 Column 680mm,ORC-BI-6C680,LBT,Biasi,"Column Radiators, Six Column Radiators",60,225,225,680,...,0,0,0,0,0,0,0,0,0,1.0
319,12675,Crane Hospital Medium 32in - High Leg,ORC-C-H-S32,Crane Hospital,Crane,"Crane, Hospital Radiators, Eton, School Radiators",67,146,146,813,...,0,0,0,0,0,0,0,0,0,1.0


In [14]:
# Save the processed Excel file
df.to_excel("output.xlsx", index=False)
print("Processing completed. Saved as output.xlsx")

Processing completed. Saved as output.xlsx


In [15]:
rows = df[(df["Ornate"] == 1) & (df["Plain"] == 1)]

display(rows)

,Products ID,Name,Castrads SKU,Family,Manufacturer,Tags,Section Length (mm),Leg Section Depth (mm),Mid Section Depth (mm),Leg Section Height (mm),...,Wall,Window,Ornate,Scroll,Round top,Flat top,Flue,Easy Clean,Plain,Nipple Top / Bottom


In [16]:
pd.set_option("display.max_colwidth", None)

In [17]:
rows = df[(df["Round top"] == 1) & (df["Flat top"] == 1)]

display(rows)

,Products ID,Name,Castrads SKU,Family,Manufacturer,Tags,Section Length (mm),Leg Section Depth (mm),Mid Section Depth (mm),Leg Section Height (mm),...,Wall,Window,Ornate,Scroll,Round top,Flat top,Flue,Easy Clean,Plain,Nipple Top / Bottom
406,11436,National Radiator Company Ideal Hospital Flat Medium 18in,ORC-NRC-IHF-M18,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators Radiators, Eton, Flat top",67,146,146,457,...,0,0,0,0,1,1,0,0,0,1.0
407,11439,National Radiator Company Ideal Hospital Flat Medium 24in,ORC-NRC-IHF-M24,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators Radiators, Eton, Flat top",67,146,146,610,...,0,0,0,0,1,1,0,0,0,1.0
408,11442,National Radiator Company Ideal Hospital Flat Medium 30in,ORC-NRC-IHF-M30,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators Radiators, Eton, Flat top",67,146,146,762,...,0,0,0,0,1,1,0,0,0,1.0
409,11445,National Radiator Company Ideal Hospital Flat Medium 36in,ORC-NRC-IHF-M36,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators Radiators, Eton, Flat top",67,146,146,914,...,0,0,0,0,1,1,0,0,0,1.0
417,979,National Radiator Company Ideal Hospital Narrow 36in,ORC-NRC-IH-N36,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators, Eton, Flat top",67,76,76,914,...,0,0,0,0,1,1,0,0,0,1.0
418,12737,National Radiator Company Ideal Hospital Wide 18in,ORC-NRC-IH-W18,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators, Eton, Flat top",67,184,184,457,...,0,0,0,0,1,1,0,0,0,1.0
419,982,National Radiator Company Ideal Hospital Wide 24in,ORC-NRC-IH-W24,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators, Eton, Flat top",67,184,184,610,...,0,0,0,0,1,1,0,0,0,1.0
420,985,National Radiator Company Ideal Hospital Wide 30in,ORC-NRC-IH-W30,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators, Eton, Flat top",67,184,184,762,...,0,0,0,0,1,1,0,0,0,1.0
421,988,National Radiator Company Ideal Hospital Wide 36in,ORC-NRC-IH-W36,Ideal Hospital,National Radiator Company,"Ideal, National Radiator Company, Hospital Radiators, Eton, Flat top",67,184,184,914,...,0,0,0,0,1,1,0,0,0,1.0


this is a dataset issue that we pointed out to our client, they said they'd have to resolve this in the future. 

In [19]:
# Read Excel
file_path2 = "output.xlsx"
df2 = pd.read_excel(file_path2)

In [20]:
# Count missing values in each column
missing_per_column = df2.isnull().sum()

print(missing_per_column)

Products ID                           0
Name                                  0
Castrads SKU                          0
Family                                0
Manufacturer                          0
Tags                                  1
Section Length (mm)                   0
Leg Section Depth (mm)                0
Mid Section Depth (mm)                0
Leg Section Height (mm)               0
Mid Section Height (mm)               0
Leg Section Weight (kg)               7
Mid Section Weight (kg)               7
Internal Volume (L)                   1
Nipple Size Top (")                   7
Nipple Size Bottom (")                7
Pipe Centre Top To Floor (mm)         2
Pipe Centre Bottom To Floor (mm)      0
Inter Axis Distance (mm)              2
Exponent N                            0
Factor Km                             0
Output Dt50 (W)                       3
Cross-brace type                    311
Top shape                             0
Link to google drive image          188


In [21]:
pd.set_option("display.max_columns", None)

In [22]:
rows = df2[df2["Leg Section Weight (kg)"].isnull()]

display(rows)

,Products ID,Name,Castrads SKU,Family,Manufacturer,Tags,Section Length (mm),Leg Section Depth (mm),Mid Section Depth (mm),Leg Section Height (mm),Mid Section Height (mm),Leg Section Weight (kg),Mid Section Weight (kg),Internal Volume (L),"Nipple Size Top ("")","Nipple Size Bottom ("")",Pipe Centre Top To Floor (mm),Pipe Centre Bottom To Floor (mm),Inter Axis Distance (mm),Exponent N,Factor Km,Output Dt50 (W),Cross-brace type,Top shape,Link to google drive image,DOP No.,Comments,Column,Row,Wall,Window,Ornate,Scroll,Round top,Flat top,Flue,Easy Clean,Plain,Nipple Top / Bottom
63,12693,American Radiator Company Detroit Ornamental 2 Column 39in,ORC-ARC-DO-2C39,National,American Radiator Company,"ARC, Detroit, Ornate",57,220,200,980,910,NaN,NaN,NaN,1.5,2.0,928.0,118,810.0,1.25,1.1100,NaN,NaN,Scrolled,NaN,202604-1064,NaN,2.0,NaN,0,0,1,1,0,0,0,0,0,0.75
112,12760,American Radiator Company Perfection 2 Column 20in,ORC-ARC-PF-2C20,Perfection,American Radiator Company,"ARC, National,Ornate Radiators, Ribbon, 2 Column",64,235,184,508,457,NaN,NaN,1.6,1.5,1.5,476.0,114,362.0,1.25,0.5867,78.0,NaN,Scrolled,NaN,202604-1113,NaN,2.0,NaN,0,0,1,1,0,0,0,0,0,1.00
114,12763,American Radiator Company Perfection 2 Column 26in,ORC-ARC-PF-2C26,Perfection,American Radiator Company,"ARC, National,Ornate Radiators, Ribbon, 2 Column",64,235,184,660,609,NaN,NaN,2.2,1.5,1.5,628.0,114,514.0,1.25,0.7521,100.0,NaN,Scrolled,NaN,202604-1115,NaN,2.0,NaN,0,0,1,1,0,0,0,0,0,1.00
115,12766,American Radiator Company Perfection 2 Column 32in,ORC-ARC-PF-2C32,Perfection,American Radiator Company,"ARC, National,Ornate Radiators, Ribbon, 2 Column",64,235,184,813,762,NaN,NaN,2.7,1.5,1.5,774.0,114,660.0,1.25,0.9402,125.0,NaN,Scrolled,NaN,202604-1116,NaN,2.0,NaN,0,0,1,1,0,0,0,0,0,1.00
116,12769,American Radiator Company Perfection 2 Column 38in,ORC-ARC-PF-2C38,Perfection,American Radiator Company,"ARC, National,Ornate Radiators, Ribbon, 2 Column",64,235,184,965,914,NaN,NaN,3.1,1.5,1.5,917.0,114,803.0,1.25,1.0910,145.0,NaN,Scrolled,NaN,202604-1117,NaN,2.0,NaN,0,0,1,1,0,0,0,0,0,1.00
117,12772,American Radiator Company Perfection 2 Column 45in,ORC-ARC-PF-2C45,Perfection,American Radiator Company,"ARC, National,Ornate Radiators, Ribbon, 2 Column",64,235,184,1143,1092,NaN,NaN,3.5,1.5,1.5,1106.0,114,992.0,1.25,1.2790,170.0,NaN,Scrolled,NaN,202604-1118,NaN,2.0,NaN,0,0,1,1,0,0,0,0,0,1.00
165,12775,American Radiator Company Rococo Window 20in,ORC-ARC-RW-20,ARC Rococo,American Radiator Company,"Window Radiators, Churchill Radiators, Warehouse Radiators",76,318,318,508,508,NaN,NaN,2.6,1.5,2.0,459.0,76,383.0,1.25,0.6393,85.0,NaN,Flat,https://drive.google.com/file/d/1J_mnxjpeKl6B3hojXr2BlCdvyc4c1Ctf/view?usp=sharing,202604-1166,NaN,NaN,NaN,0,1,0,0,0,1,0,0,0,0.75


same rows are missing for leg section weight and mid section weight. 

In [23]:
rows = df2[df2['Nipple Size Top (")'].isnull()]

display(rows)

,Products ID,Name,Castrads SKU,Family,Manufacturer,Tags,Section Length (mm),Leg Section Depth (mm),Mid Section Depth (mm),Leg Section Height (mm),Mid Section Height (mm),Leg Section Weight (kg),Mid Section Weight (kg),Internal Volume (L),"Nipple Size Top ("")","Nipple Size Bottom ("")",Pipe Centre Top To Floor (mm),Pipe Centre Bottom To Floor (mm),Inter Axis Distance (mm),Exponent N,Factor Km,Output Dt50 (W),Cross-brace type,Top shape,Link to google drive image,DOP No.,Comments,Column,Row,Wall,Window,Ornate,Scroll,Round top,Flat top,Flue,Easy Clean,Plain,Nipple Top / Bottom
311,10835,Church 20in,ORC-C-20,Church,Unknown,Church,76,102,102,508,508,2.4,2.4,0.33,NaN,NaN,32.0,32,0.0,1.25,0.53,70.0,NaN,Flat,NaN,202604-1312,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN
312,10838,Church 26in,ORC-C-26,Church,Unknown,Church,76,102,102,660,660,3.1,3.1,0.43,NaN,NaN,32.0,32,0.0,1.25,0.68,91.0,NaN,Flat,NaN,202604-1313,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN
313,10841,Church 32in,ORC-C-32,Church,Unknown,Church,76,102,102,813,813,3.8,3.8,0.53,NaN,NaN,32.0,32,0.0,1.25,0.84,112.0,NaN,Flat,https://drive.google.com/file/d/1NfRVbU0F4o4_RVa4U_n7ZF0a5S-21VEH/view?usp=sharing,202604-1314,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN
314,10844,Church 36in,ORC-C-36,Church,Unknown,Church,76,102,102,914,914,4.2,4.2,0.60,NaN,NaN,32.0,32,0.0,1.25,0.95,126.0,NaN,Flat,NaN,202604-1315,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN
315,10847,Church 38in,ORC-C-38,Church,Unknown,Church,76,102,102,965,965,4.5,4.5,0.63,NaN,NaN,32.0,32,0.0,1.25,1.00,133.0,NaN,Flat,https://drive.google.com/file/d/1Dn6ObAZqcoZSYAPio0Gq3khKmITT86rb/view?usp=sharing,202604-1316,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN
442,10829,Wave 29in H x 40in W,ORC-W-29-40,Church,Unknown,"Wave, Church",76,178,178,737,737,70.0,70.0,15.00,NaN,NaN,32.0,32,0.0,1.25,6.90,918.0,NaN,Flat,https://drive.google.com/file/d/1fBuHBoLNCqmQgJCKFDgyxt55Sq7yw0Gu/view?usp=sharing,202604-1443,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN
443,10832,Wave 37in H x 40in W,ORC-W-37-40,Church,Unknown,"Wave, Church",76,178,102,940,940,90.0,90.0,19.00,NaN,NaN,32.0,32,0.0,1.25,8.81,1172.0,NaN,Flat,NaN,202604-1444,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,NaN


same rows are missing for nipple size data 